# Phase 7 — Self-Supervised Pretraining (no EF labels used)

**What 'semi-supervised' means here, concretely:** EchoNet-Dynamic has an EF label for every video, so there's no separate 'unlabeled pool' the way TMED-2 had. Instead, we get the semi/self-supervised benefit a different way: pretrain the ResNet18 backbone using a **self-supervised contrastive objective (SimCLR-style)** on frames sampled from all videos, **without ever using the EF labels**, then fine-tune the already-built heteroscedastic model (notebook 05) starting from these pretrained weights instead of plain ImageNet weights.

**Why this is a legitimate 'semi-supervised' framing for the paper:** the backbone learns general echo-image structure from unlabeled views (two random augmentations of the same frame should have similar representations). The *label-efficiency* question (how much does the model improve with fewer labeled examples) is still testable: fine-tune this pretrained backbone on a fraction of the labels and see if it needs less labeled data than ImageNet-only init to reach the same accuracy.

Reuses the frame cache from notebook 04 — no new data needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, glob, json, time, random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.models as models
import torchvision.transforms as T
from torch.utils.data import Dataset, DataLoader

PROJECT_ROOT = '/content/drive/MyDrive/echonet_project'
DATA_DIR = os.path.join(PROJECT_ROOT, 'data/raw', 'EchoNet-Dynamic', 'EchoNet-Dynamic')
FULL_CSV = os.path.join(DATA_DIR, 'FileList.csv')
FRAME_CACHE_DIR = os.path.join(PROJECT_ROOT, 'data/full', 'frames')

df = pd.read_csv(FULL_CSV)
filename_col = 'FileName' if 'FileName' in df.columns else [c for c in df.columns if 'file' in c.lower() and 'name' in c.lower()][0]

valid_mask = df[filename_col].apply(lambda f: os.path.exists(
    os.path.join(FRAME_CACHE_DIR, (str(f) if str(f).endswith('.avi') else str(f)+'.avi').replace('.avi', '.npy'))))
clean_df = df[valid_mask].reset_index(drop=True)
# Use ALL splits (train+val+test frames) for pretraining -- legitimate because we never touch
# the EF labels here, only raw pixels. Held-out test images are never used for anything supervised.
print(f'Frames available for self-supervised pretraining: {len(clean_df)}')

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', device)
if device == 'cpu':
    print('WARNING: enable GPU -- contrastive pretraining is expensive on CPU.')

## SimCLR-style contrastive dataset: two augmented views per frame, no labels

Standard, lightweight augmentations appropriate for grayscale-ish ultrasound: random crop+resize, horizontal flip, mild brightness/contrast jitter. We deliberately avoid color jitter beyond brightness/contrast (irrelevant for near-grayscale echo) and avoid aggressive rotation (echo views have a meaningful, roughly-fixed orientation).

In [ ]:
class SimCLRFrameDataset(Dataset):
    def __init__(self, df, frame_dir, filename_col):
        self.df = df.reset_index(drop=True)
        self.frame_dir = frame_dir
        self.filename_col = filename_col
        mean, std = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]
        self.aug = T.Compose([
            T.ToPILImage(),
            T.RandomResizedCrop(112, scale=(0.6, 1.0)),
            T.RandomHorizontalFlip(),
            T.ColorJitter(brightness=0.3, contrast=0.3),
            T.ToTensor(),
            T.Normalize(mean, std),
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        fname = str(row[self.filename_col])
        if not fname.endswith('.avi'):
            fname = fname + '.avi'
        frame = np.load(os.path.join(self.frame_dir, fname.replace('.avi', '.npy')))
        view1 = self.aug(frame)
        view2 = self.aug(frame)
        return view1, view2

BATCH_SIZE = 64  # contrastive learning benefits from larger batches; lower if you hit OOM
ssl_ds = SimCLRFrameDataset(clean_df, FRAME_CACHE_DIR, filename_col)
ssl_loader = DataLoader(ssl_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, drop_last=True)
print('SSL dataset ready:', len(ssl_ds), 'frames,', len(ssl_loader), 'batches/epoch')

## SimCLR model: ResNet18 backbone + projection head, NT-Xent contrastive loss

In [ ]:
class SimCLRModel(nn.Module):
    def __init__(self, proj_dim=128):
        super().__init__()
        backbone = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
        n_feat = backbone.fc.in_features
        backbone.fc = nn.Identity()
        self.backbone = backbone
        self.projector = nn.Sequential(
            nn.Linear(n_feat, n_feat), nn.ReLU(inplace=True), nn.Linear(n_feat, proj_dim)
        )

    def forward(self, x):
        feat = self.backbone(x)
        proj = F.normalize(self.projector(feat), dim=1)
        return proj

def nt_xent_loss(z1, z2, temperature=0.5):
    """Standard SimCLR contrastive loss: pull together the two views of the same image,
    push apart all other images in the batch."""
    N = z1.size(0)
    z = torch.cat([z1, z2], dim=0)  # (2N, D)
    sim = torch.mm(z, z.t()) / temperature  # (2N, 2N)
    sim.fill_diagonal_(-1e9)  # mask self-similarity
    targets = torch.cat([torch.arange(N, 2 * N), torch.arange(0, N)]).to(z.device)
    return F.cross_entropy(sim, targets)

ssl_model = SimCLRModel().to(device)
ssl_optimizer = torch.optim.Adam(ssl_model.parameters(), lr=3e-4)

## Checkpointing + train (time-based, same pattern as notebooks 04/05)

In [ ]:
EXP_NAME = 'simclr_pretrain_v1'
CHECKPOINT_EVERY_MINUTES = 15
STATE_PATH = os.path.join(PROJECT_ROOT, 'checkpoints', EXP_NAME, 'latest.pt')
os.makedirs(os.path.dirname(STATE_PATH), exist_ok=True)

def save_state(model, optimizer, epoch, step_in_epoch, note=''):
    torch.save({'epoch': epoch, 'step_in_epoch': step_in_epoch,
                'model_state': model.state_dict(), 'optimizer_state': optimizer.state_dict()}, STATE_PATH)
    print(f'[checkpoint] {note} epoch={epoch} step={step_in_epoch}')

def load_state(model, optimizer):
    if not os.path.exists(STATE_PATH):
        print('[checkpoint] none found, starting from scratch')
        return 0, 0
    state = torch.load(STATE_PATH, map_location=device)
    model.load_state_dict(state['model_state'])
    optimizer.load_state_dict(state['optimizer_state'])
    print(f"[checkpoint] resumed from epoch={state['epoch']} step={state['step_in_epoch']}")
    return state['epoch'], state['step_in_epoch']

start_epoch, resume_step = load_state(ssl_model, ssl_optimizer)
N_EPOCHS_SSL = 10  # self-supervised pretraining typically wants more epochs than supervised fine-tuning
last_checkpoint_time = time.time()

for epoch in range(start_epoch, N_EPOCHS_SSL):
    ssl_model.train()
    running_loss, n_batches = 0.0, 0
    for step, (v1, v2) in enumerate(ssl_loader):
        if epoch == start_epoch and step < resume_step:
            continue
        v1, v2 = v1.to(device), v2.to(device)
        z1, z2 = ssl_model(v1), ssl_model(v2)
        loss = nt_xent_loss(z1, z2)
        ssl_optimizer.zero_grad(); loss.backward(); ssl_optimizer.step()
        running_loss += loss.item(); n_batches += 1
        if (time.time() - last_checkpoint_time) > CHECKPOINT_EVERY_MINUTES * 60:
            save_state(ssl_model, ssl_optimizer, epoch, step + 1, note='mid-epoch')
            last_checkpoint_time = time.time()
    resume_step = 0
    print(f'Epoch {epoch}: contrastive_loss={running_loss/max(n_batches,1):.3f}')
    save_state(ssl_model, ssl_optimizer, epoch + 1, 0, note='epoch-end')

print('SSL pretraining complete (or reached N_EPOCHS_SSL). Re-run to resume after a disconnect.')
print('A healthy run shows contrastive_loss decreasing steadily, typically from ~log(2*batch-1) down toward 1-3.')

## Export the pretrained backbone weights (for use in fine-tuning, notebook 08)

In [ ]:
backbone_path = os.path.join(PROJECT_ROOT, 'checkpoints', EXP_NAME, 'backbone_only.pt')
torch.save(ssl_model.backbone.state_dict(), backbone_path)
print('Saved pretrained backbone weights to', backbone_path)
print('Use this in notebook 08 as the starting point for the heteroscedastic model, instead of plain ImageNet weights.')

## What to check before moving on

- `contrastive_loss` should decrease steadily across epochs. If it plateaus very early or stays flat, something is likely wrong (e.g., augmentations too weak/strong, batch size too small for good negatives) -- flag this rather than proceeding.
- This notebook only pretrains the backbone; it does **not** touch EF or HFrEF labels at any point, keeping the self-supervised claim honest.
- Next step (notebook 08): re-run the heteroscedastic fine-tuning (same as notebook 05) but initialize `self.backbone` from `backbone_only.pt` instead of `ResNet18_Weights.IMAGENET1K_V1`, and compare final val_mae/val_auc against the notebook 05 numbers -- ideally also repeat this comparison at reduced label fractions (e.g. 25%, 50% of TRAIN) to test the label-efficiency claim from the roadmap's novelty audit.